# Acne Detection

## Install YOLO11 via Ultralytics

In [ ]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import os

# Lihat isi folder saat ini
print(os.listdir())

# Cek apakah file yaml ada
print(os.path.exists('/content/drive/MyDrive/AIOT/acne-dataset/data.yaml'))

In [ ]:
from ultralytics import YOLO
import time

# Load YOLOv8 model (bisa ganti ke yolov8s.yaml atau lainnya)
model = YOLO('yolov8s.yaml')  # n = nano, ringan untuk awal

start = time.time()

# Train model
model.train(
    data='/content/drive/MyDrive/AIOT/acne-dataset/data.yaml',
    epochs=100,                # bisa 50 kalau waktu terbatas
    imgsz=640,                 # gambar lebih besar = deteksi lebih detail
    batch=16,                  # aman untuk GPU Colab free
    name='acne_yolov8s',
)

end = time.time()
print(f"Total training time: {(end - start) / 60:.2f} minutes")

!cp /content/runs/detect/acne_yolov8s/weights/best.pt /content/drive/MyDrive/AIOT/
print("Model berhasil disalin ke Google Drive!")

## Evaluasi Model

In [ ]:
from ultralytics import YOLO

# Load model hasil training
model = YOLO('/content/drive/MyDrive/AIOT/best.pt')

# Evaluasi model di validasi set
metrics = model.val(data='/content/drive/MyDrive/AIOT/acne-dataset/data.yaml')

# Tampilkan metrik utama
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"Precision (mean): {metrics.box.mp:.4f}")
print(f"Recall (mean): {metrics.box.mr:.4f}")


- `mAP50-95	= 0.2831` <br>Mean Average Precision pada IoU threshold dari 0.5 sampai 0.95 (interval 0.05). Ini adalah metrik utama untuk menilai kualitas deteksi. Nilai ini dianggap standar COCO, makin tinggi makin bagus. Saat model diuji dengan berbagai tingkat ketelitian, performa turun jadi sedang. Ini menandakan bahwa bounding box dari model kurang presisi atau konsisten. Akurasi deteksi berkurang saat butuh presisi tinggi (mAP50-95 hanya 28.31%)
- `mAP50	= 0.6385`  <br>	Average Precision pada IoU = 0.5. Metrik ini lebih toleran, dan biasanya nilainya lebih tinggi. Tapi tidak seketat mAP50-95. deteksi cukup bagus saat toleransi posisi bounding box longgar.
- `Precision = 61.3%` → dari semua jerawat yang diprediksi oleh model, 61.3% benar-benar jerawat.
- `Recall = 61.6%` → dari semua jerawat yang ada di gambar, 61.6% berhasil ditemukan oleh model.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

# Ambil semua file gambar evaluasi
image_paths = glob.glob('runs/detect/val14/*.png')

# Tampilkan semua gambar
for path in image_paths:
    img = mpimg.imread(path)
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.title(path.split('/')[-1])
    plt.axis('off')
    plt.show()


In [ ]:
# Ambil semua file gambar evaluasi
image_paths = glob.glob('runs/detect/val14/*.jpg')

# Tampilkan semua gambar
for path in image_paths:
    img = mpimg.imread(path)
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.title(path.split('/')[-1])
    plt.axis('off')
    plt.show()

## Test Model

In [ ]:
from ultralytics import YOLO

# Load model hasil training
model = YOLO('/content/drive/MyDrive/AIOT/best.pt')
results = model.predict(source='/content/drive/MyDrive/AIOT/acne-dataset/test/images/acne-102_jpeg.rf.d15b34dc9a1e69aa996118a4c1a173ef.jpg', save=True, conf=0.3)